# Semana 7 — Cohorte, *features* sin fuga temporal y modelo de reingreso a 30 días

**Objetivo:** desarrollar y validar internamente un modelo que estime, al alta hospitalaria, el riesgo de **reingreso no programado o muerte en los 30 días siguientes**, el mismo desenlace del índice LACE (van Walraven et al., CMAJ 2010). Lo construyes sobre las tablas OMOP de la semana 6 y lo reportas siguiendo la lógica de TRIPOD+AI.

**Productos reales usados:** OMOP CDM, DuckDB, scikit-learn.

**Al terminar tendrás:** `data/models/readmission_model.joblib`, `data/reports/s07_metrics.json` y `data/reports/model_card_s07.md`.

**Requisitos:** semanas 5 y 6.

### Celda de configuración del entorno (se repite al inicio de cada semana)

**Explicación línea por línea**

- `import os`, `import sys`, `from pathlib import Path`: cargan tres módulos de la librería estándar. `os` lee variables de entorno, `sys` informa la versión de Python y `Path` representa rutas de archivos como objetos. **Por qué así:** `Path` evita concatenar rutas con texto (`"a" + "/" + "b"`), que falla entre sistemas operativos. **En la práctica real** todo código de datos moderno usa `pathlib`.
- `try: import google.colab` … `except ImportError`: intenta importar un módulo que solo existe en Colab. Si la importación funciona, `IN_COLAB = True`; si lanza `ImportError`, estamos en otra máquina. **Por qué importa:** el mismo notebook corre en Colab y en tu laptop sin editarlo. **Cuándo se usa:** siempre que un código deba ser portable entre entornos.
- `from google.colab import drive` y `drive.mount("/content/drive")`: conectan tu Google Drive al sistema de archivos de Colab. **Por qué:** el disco de Colab se borra al cerrar la sesión; en Drive los datos de la semana 1 siguen ahí el lunes siguiente. La primera vez Colab te pedirá autorizar el acceso.
- `PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")`: carpeta raíz del proyecto dentro de tu Drive.
- `Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()`: fuera de Colab usa la variable de entorno `HEALTHETL_PROJECT_DIR` si existe; si no, la carpeta actual (`"."`). `.resolve()` la convierte en ruta absoluta. **En la práctica real** la configuración se pasa por variables de entorno, nunca escrita a mano dentro del código (principio *12-factor app*).
- `DATA_DIR = PROJECT_DIR / "data"`: el operador `/` de `pathlib` une rutas de forma segura.
- `DATA_DIR.mkdir(parents=True, exist_ok=True)`: crea la carpeta y sus padres; `exist_ok=True` evita un error si ya existe. Esto hace la celda **idempotente**: puedes ejecutarla 1 o 10 veces con el mismo resultado, una propiedad central de todo pipeline ETL.
- Los tres `print`: confirman versión de Python, entorno y ruta. Registrar el entorno al inicio es un hábito de reproducibilidad.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab  # este módulo solo existe dentro de Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/health-data-etl-lab")
else:
    PROJECT_DIR = Path(os.environ.get("HEALTHETL_PROJECT_DIR", ".")).resolve()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("¿Colab?:", IN_COLAB)
print("Carpeta de datos:", DATA_DIR)

### Paso 1 — Librerías y tablas OMOP

**Explicación línea por línea**

- `import numpy as np`: cálculo numérico (bootstrap, vectores).
- `from sklearn... import ...`: `Pipeline`/`make_pipeline` encadenan pasos; `SimpleImputer` imputa ausentes; `StandardScaler` estandariza; `LogisticRegression` y `HistGradientBoostingClassifier` son los dos modelos; `StratifiedGroupKFold` y `StratifiedKFold` son esquemas de validación cruzada; `roc_auc_score` y `brier_score_loss` son métricas.
- `for t in [...]: con.execute(f"CREATE VIEW {t} AS SELECT * FROM read_parquet(...)")`: expone las 6 tablas OMOP.
- `CREATE VIEW concept_sets ...`: los conjuntos de conceptos de la semana 5.
- `SEED = 42`: semilla fija para que cada ejecución dé los mismos resultados.

In [ ]:
%pip install -q "duckdb>=1.1"

In [ ]:
import json

import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

OMOP_DIR = DATA_DIR / "omop"
REPORTS_DIR = DATA_DIR / "reports"
MODELS_DIR = DATA_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")
for t in ["person", "observation_period", "visit_occurrence", "condition_occurrence", "measurement", "death"]:
    con.execute(f"CREATE OR REPLACE VIEW {t} AS SELECT * FROM read_parquet('{(OMOP_DIR / f'{t}.parquet').as_posix()}')")
con.execute(f"CREATE OR REPLACE VIEW concept_sets AS SELECT * FROM read_parquet('{(DATA_DIR / 'reference' / 'concept_sets.parquet').as_posix()}')")
con.sql("SELECT concept_set, COUNT(*) AS codigos, MIN(source) AS fuente FROM concept_sets GROUP BY 1").df()

### Paso 2 — Diseño de la cohorte (antes de escribir código)

Un modelo de predicción se define por **cuándo** se predice (fecha índice), **sobre quién** (criterios de inclusión), **qué** (desenlace) y **en qué ventana** (horizonte).

- **Unidad de análisis:** cada hospitalización (`visit_concept_id = 9201`). Una persona puede aportar varias: lo trataremos en la validación.
- **Fecha índice:** el alta (`visit_end_datetime`). Solo se puede usar información disponible **hasta ese momento**.
- **Desenlace (como LACE):** nueva hospitalización que empieza en los 30 días posteriores al alta **o** muerte en esos 30 días. LACE combina ambos para evitar el sesgo de "censurar" a quien muere antes de poder reingresar.
- **Exclusiones:** muertes durante la hospitalización (no hubo alta con vida) y altas sin 30 días de seguimiento posibles sin evento.
- **Periodo:** altas desde el 1-ene-2012. **Por qué:** Synthea exporta por defecto solo 10 años de historia detallada (`exporter.years_of_history = 10` en `synthea.properties`) y esta muestra se generó en noviembre de 2021; antes de esa fecha faltan observaciones y diagnósticos, y las *features* serían artificialmente vacías.

### Paso 3 — Construir la cohorte en SQL

**Explicación línea por línea (SQL)**

- `WITH ip AS (...)`: hospitalizaciones con la fecha de inicio de la **siguiente** hospitalización de la misma persona (`LEAD`).
- `date_diff('hour', ...) / 24.0 AS los_days`: duración de la estancia en días con decimales (componente "L" de LACE).
- `CASE WHEN ip.next_start <= ip.visit_end_datetime + INTERVAL 30 DAY THEN 1 ELSE 0 END AS readmit_30d`: reingreso en 30 días.
- `CASE WHEN d.death_date > ip.visit_end_date AND d.death_date <= ip.visit_end_date + INTERVAL 30 DAY ...`: muerte en los 30 días posteriores.
- `GREATEST(readmit_30d, death_30d) AS y`: desenlace compuesto (1 si ocurre cualquiera).
- `WHERE ip.visit_end_date >= DATE '2012-01-01'`: periodo con historia completa.
- `AND (d.death_date IS NULL OR d.death_date > ip.visit_end_date)`: excluye muertes durante la estancia.
- `AND (op.observation_period_end_date >= ip.visit_end_date + INTERVAL 30 DAY OR y = 1)`: exige seguimiento suficiente si no hubo evento (si no, un "0" podría ser solo falta de datos).

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE cohort AS
WITH ip AS (
    SELECT *, LEAD(visit_start_datetime) OVER (PARTITION BY person_id ORDER BY visit_start_datetime) AS next_start
    FROM visit_occurrence
    WHERE visit_concept_id = 9201
),
base AS (
    SELECT ip.visit_occurrence_id, ip.person_id,
           ip.visit_start_datetime AS admit_ts, ip.visit_end_datetime AS discharge_ts,
           ip.visit_end_date AS discharge_date,
           date_diff('hour', ip.visit_start_datetime, ip.visit_end_datetime) / 24.0 AS los_days,
           CASE WHEN ip.next_start <= ip.visit_end_datetime + INTERVAL 30 DAY THEN 1 ELSE 0 END AS readmit_30d,
           CASE WHEN d.death_date > ip.visit_end_date
                 AND d.death_date <= ip.visit_end_date + INTERVAL 30 DAY THEN 1 ELSE 0 END AS death_30d,
           d.death_date, op.observation_period_end_date
    FROM ip
    JOIN observation_period op USING (person_id)
    LEFT JOIN death d USING (person_id)
)
SELECT *, GREATEST(readmit_30d, death_30d) AS y
FROM base
WHERE discharge_date >= DATE '2012-01-01'
  AND (death_date IS NULL OR death_date > discharge_date)
  AND (observation_period_end_date >= discharge_date + INTERVAL 30 DAY OR GREATEST(readmit_30d, death_30d) = 1)
""")
con.sql("""
SELECT COUNT(*) AS altas, COUNT(DISTINCT person_id) AS personas,
       SUM(y) AS eventos, SUM(readmit_30d) AS reingresos, SUM(death_30d) AS muertes,
       ROUND(AVG(y), 3) AS tasa
FROM cohort
""").df()

### Paso 4 — *Features* calculadas solo con información previa al alta

La **fuga temporal** (*temporal leakage*) ocurre cuando una variable usa información posterior al momento de la predicción. El modelo parece excelente en el laboratorio y falla en producción. Cada subconsulta de abajo tiene un límite temporal explícito.

**Explicación línea por línea (SQL)**

- `age_at_discharge`: edad al alta, a partir de `birth_datetime`.
- `male`: 1 si `gender_concept_id = 8507`.
- `ed_visits_6m`: urgencias en los 6 meses **antes del ingreso** (`< c.admit_ts`): componente "E" de LACE.
- `ip_visits_12m`: hospitalizaciones previas en 12 meses; predictor clásico de reingreso.
- `n_active_conditions`: condiciones distintas activas al alta (empezaron en o antes del alta y no habían terminado): aproximación a la carga de comorbilidad ("C" de LACE usa el índice de Charlson, que requiere códigos CIE que Synthea no trae).
- `has_htn`, `has_t2dm_or_prediabetes`: comorbilidades definidas con los **conjuntos de conceptos** de la semana 5 (`IN (SELECT code FROM concept_sets WHERE ...)`).
- `last_sbp`, `last_bmi`: último valor en los 3 años previos al alta (`measurement_datetime <= c.discharge_ts`). `QUALIFY ROW_NUMBER() ... = 1` toma el más reciente.
- `LEFT JOIN`: conserva altas sin mediciones; quedarán como ausentes y se imputarán dentro del modelo.
- El componente "A" de LACE (ingreso urgente) no se construye: Synthea no enlaza la urgencia con la hospitalización que la siguió. Queda documentado como limitación.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE features AS
WITH meas AS (
    SELECT c.visit_occurrence_id, m.measurement_source_value AS code, m.value_as_number
    FROM cohort c
    JOIN measurement m ON m.person_id = c.person_id
     AND m.measurement_datetime <= c.discharge_ts
     AND m.measurement_datetime > c.discharge_ts - INTERVAL 3 YEAR
    WHERE m.measurement_source_value IN ('8480-6', '39156-5')
    QUALIFY ROW_NUMBER() OVER (PARTITION BY c.visit_occurrence_id, m.measurement_source_value
                               ORDER BY m.measurement_datetime DESC) = 1
)
SELECT c.visit_occurrence_id, c.person_id, c.discharge_date, c.y,
       date_diff('day', p.birth_datetime::DATE, c.discharge_date) / 365.25 AS age_at_discharge,
       CASE WHEN p.gender_concept_id = 8507 THEN 1 ELSE 0 END AS male,
       c.los_days,
       (SELECT COUNT(*) FROM visit_occurrence v WHERE v.person_id = c.person_id AND v.visit_concept_id = 9203
           AND v.visit_start_datetime < c.admit_ts AND v.visit_start_datetime >= c.admit_ts - INTERVAL 6 MONTH) AS ed_visits_6m,
       (SELECT COUNT(*) FROM visit_occurrence v WHERE v.person_id = c.person_id AND v.visit_concept_id = 9201
           AND v.visit_start_datetime < c.admit_ts AND v.visit_start_datetime >= c.admit_ts - INTERVAL 12 MONTH) AS ip_visits_12m,
       (SELECT COUNT(DISTINCT co.condition_source_value) FROM condition_occurrence co WHERE co.person_id = c.person_id
           AND co.condition_start_date <= c.discharge_date
           AND (co.condition_end_date IS NULL OR co.condition_end_date > c.discharge_date)) AS n_active_conditions,
       (SELECT COUNT(*) > 0 FROM condition_occurrence co WHERE co.person_id = c.person_id
           AND co.condition_start_date <= c.discharge_date
           AND co.condition_source_value IN (SELECT code FROM concept_sets WHERE concept_set = 'hypertension'))::INTEGER AS has_htn,
       (SELECT COUNT(*) > 0 FROM condition_occurrence co WHERE co.person_id = c.person_id
           AND co.condition_start_date <= c.discharge_date
           AND co.condition_source_value IN (SELECT code FROM concept_sets WHERE concept_set = 't2dm_or_prediabetes'))::INTEGER AS has_t2dm_or_prediabetes,
       MAX(CASE WHEN m.code = '8480-6' THEN m.value_as_number END) AS last_sbp,
       MAX(CASE WHEN m.code = '39156-5' THEN m.value_as_number END) AS last_bmi
FROM cohort c
JOIN person p USING (person_id)
LEFT JOIN meas m USING (visit_occurrence_id)
GROUP BY ALL
""")
df = con.sql("SELECT * FROM features ORDER BY visit_occurrence_id").df()
print(df.shape)
df.head()

### Paso 5 — Descripción de la cohorte y tamaño muestral

**Explicación línea por línea**

- `FEATURES = [...]`: lista explícita de predictores. **Nunca** uses "todas las columnas": así evitas meter por accidente identificadores o el propio desenlace.
- `df.groupby("y")[FEATURES].mean().T.round(2)`: la "Tabla 1" de todo artículo clínico: medias por grupo de desenlace. `.T` transpone para leer una variable por fila.
- `df[FEATURES].isna().mean()`: proporción de ausentes por predictor; TRIPOD+AI pide reportarlo.
- `epv = n_eventos / len(FEATURES)`: eventos por variable candidata. La regla clásica pide al menos 10; Riley et al. (BMJ 2020) proponen un cálculo de tamaño muestral más riguroso, pero el EPV sirve como primer control.
- `n_altas_por_persona`: cuántas altas aporta cada persona; si una persona aporta muchas, sus observaciones están **correlacionadas**.

In [ ]:
FEATURES = ["age_at_discharge", "male", "los_days", "ed_visits_6m", "ip_visits_12m",
            "n_active_conditions", "has_htn", "has_t2dm_or_prediabetes", "last_sbp", "last_bmi"]

n_eventos = int(df["y"].sum())
print(f"Altas: {len(df)} | personas: {df['person_id'].nunique()} | eventos: {n_eventos} ({df['y'].mean():.1%})")
print(f"Eventos por variable (EPV): {n_eventos / len(FEATURES):.1f}")
n_altas_por_persona = df.groupby("person_id").size()
print(f"Altas por persona: mediana {n_altas_por_persona.median():.0f}, máximo {n_altas_por_persona.max()}")
print("\n% ausentes por predictor:")
print((df[FEATURES].isna().mean() * 100).round(1).to_string())
df.groupby("y")[FEATURES].mean().T.round(2)

### Paso 6 — Modelos dentro de un `Pipeline`

**Explicación línea por línea**

- `X = df[FEATURES]`, `y = df["y"].to_numpy()`, `groups = df["person_id"].to_numpy()`: predictores, desenlace y grupo (persona).
- `make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(), LogisticRegression(max_iter=2000))`:
    - `SimpleImputer(strategy="median")` reemplaza ausentes por la mediana; `add_indicator=True` añade una columna 0/1 "estaba ausente". **Por qué:** en historias clínicas que falte una medición es información (recuerda a Agniel et al.).
    - `StandardScaler()` pone cada variable con media 0 y desviación 1, necesario para que la regresión penalizada trate igual a todas.
    - `LogisticRegression(max_iter=2000)`: modelo base interpretable (el mismo tipo de modelo con el que se construyó LACE); `max_iter` evita avisos de no convergencia.
- **Por qué todo dentro de un pipeline:** la mediana y la escala se calculan **solo con los datos de entrenamiento** de cada partición. Calcularlas antes con todos los datos es otra forma de fuga.
- `HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED)`: árboles potenciados; maneja ausentes de forma nativa y capta no linealidades. Profundidad baja y tasa de aprendizaje pequeña reducen el sobreajuste en muestras chicas.

In [ ]:
X = df[FEATURES]
y = df["y"].to_numpy()
groups = df["person_id"].to_numpy()

MODELS = {
    "logistic": make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                              LogisticRegression(max_iter=2000)),
    "gbm": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=SEED),
}
list(MODELS)

### Paso 7 — Demostración de fuga entre pacientes: validación ingenua vs. agrupada

**Explicación línea por línea**

- `naive = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)`: reparte **altas** al azar; la misma persona puede quedar en entrenamiento y en prueba.
- `grouped = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)`: reparte **personas**: todas las altas de una persona caen en la misma partición, y mantiene la proporción de eventos lo más parecida posible.
- `cross_val_predict(model, X, y, cv=cv, groups=..., method="predict_proba")[:, 1]`: entrena 5 veces y devuelve, para cada alta, la probabilidad predicha por el modelo que **no** la vio (predicción *out-of-fold*). `[:, 1]` toma la probabilidad de la clase 1.
- `roc_auc_score(y, p)`: AUROC (estadístico C): probabilidad de que una alta con evento reciba mayor riesgo que una sin evento.
- **Qué verás:** el modelo de árboles obtiene un AUROC más alto con la validación ingenua porque "reconoce" a pacientes que ya vio. Ese número es falso; el agrupado es el honesto.

In [ ]:
naive = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grouped = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)

for name, model in MODELS.items():
    for cv_name, cv in [("ingenua", naive), ("agrupada", grouped)]:
        p = cross_val_predict(model, X, y, cv=cv, groups=groups if cv_name == "agrupada" else None,
                              method="predict_proba")[:, 1]
        print(f"{name:<9} validación {cv_name:<9} AUROC = {roc_auc_score(y, p):.3f}")

### Paso 8 — Validación interna: validación cruzada agrupada repetida

Con un solo particionado, el resultado depende del azar. Se repite 5 veces con distintas semillas.

**Explicación línea por línea**

- `def oof_predictions(model, seed):` devuelve las predicciones *out-of-fold* de una repetición.
- `for rep in range(5):` cinco repeticiones; `seed=SEED + rep` cambia el reparto.
- `brier_score_loss(y, p)`: error cuadrático medio entre probabilidad y desenlace (0 = perfecto). Combina discriminación y calibración.
- `results[name] = {...}`: guarda media y rango de AUROC y Brier.
- `oof[name] = oof_predictions(model, SEED)`: conserva las predicciones de la primera repetición para los pasos siguientes.
- `brier_ref = y.mean() * (1 - y.mean())`: Brier de un modelo "nulo" que predice a todos la tasa global; tu modelo debe ser claramente menor.

In [ ]:
def oof_predictions(model, seed):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    return cross_val_predict(model, X, y, cv=cv, groups=groups, method="predict_proba")[:, 1]


results, oof = {}, {}
for name, model in MODELS.items():
    aucs, briers = [], []
    for rep in range(5):
        p = oof_predictions(model, SEED + rep)
        aucs.append(roc_auc_score(y, p))
        briers.append(brier_score_loss(y, p))
    results[name] = {"auroc_mean": float(np.mean(aucs)), "auroc_min": float(np.min(aucs)), "auroc_max": float(np.max(aucs)),
                     "brier_mean": float(np.mean(briers))}
    oof[name] = oof_predictions(model, SEED)

brier_ref = float(y.mean() * (1 - y.mean()))
print(f"Brier de referencia (modelo nulo): {brier_ref:.4f}")
pd.DataFrame(results).T.round(3)

### Paso 9 — Intervalo de confianza con *bootstrap* por paciente

Como las altas de una misma persona están correlacionadas, el remuestreo se hace por **persona** (*cluster bootstrap*), no por alta.

**Explicación línea por línea**

- `rng = np.random.default_rng(SEED)`: generador aleatorio moderno de NumPy con semilla.
- `personas = np.unique(groups)` e `idx_por_persona = {...}`: índices de las filas de cada persona.
- `for _ in range(N_BOOT):` cada iteración sortea personas **con reemplazo** (`rng.choice(..., replace=True)`) y reúne todas sus altas.
- `if y[idx].min() == y[idx].max(): continue`: si una muestra sale con un solo tipo de desenlace, el AUROC no se puede calcular; se descarta.
- `np.percentile(aucs, [2.5, 97.5])`: intervalo de confianza del 95 % por percentiles.

In [ ]:
rng = np.random.default_rng(SEED)
personas = np.unique(groups)
idx_por_persona = {g: np.flatnonzero(groups == g) for g in personas}
N_BOOT = 300

for name, p in oof.items():
    aucs = []
    for _ in range(N_BOOT):
        sample = rng.choice(personas, size=len(personas), replace=True)
        idx = np.concatenate([idx_por_persona[g] for g in sample])
        if y[idx].min() == y[idx].max():
            continue
        aucs.append(roc_auc_score(y[idx], p[idx]))
    lo, hi = np.percentile(aucs, [2.5, 97.5])
    results[name]["auroc_ci95"] = [float(lo), float(hi)]
    print(f"{name:<9} AUROC {roc_auc_score(y, p):.3f} (IC95 % bootstrap por paciente {lo:.3f}–{hi:.3f})")

### Paso 10 — Calibración: ¿el 30 % predicho es un 30 % observado?

Un modelo puede ordenar bien (AUROC alto) y aun así dar probabilidades equivocadas. TRIPOD+AI exige reportar la calibración.

**Explicación línea por línea**

- `pd.qcut(p, q=5, duplicates="drop")`: divide las predicciones en 5 grupos de igual tamaño (quintiles).
- `.groupby(...).agg(predicho=("p", "mean"), observado=("y", "mean"), n=("y", "size"))`: en cada quintil compara riesgo medio predicho y tasa observada; `observed=True` evita grupos vacíos.
- `oe = y.mean() / p.mean()`: razón observado/esperado; 1 = calibración global perfecta.
- `logit = np.log(p_c / (1 - p_c))`: transforma la probabilidad a la escala logit; `np.clip` evita `log(0)`.
- `LogisticRegression(C=1e6).fit(logit.reshape(-1, 1), y).coef_[0][0]`: **pendiente de calibración**: se ajusta una regresión del desenlace sobre el logit predicho, sin penalización práctica (`C` muy grande). 1 es ideal; menor que 1 indica predicciones demasiado extremas (sobreajuste).

In [ ]:
for name, p in oof.items():
    tab = (pd.DataFrame({"p": p, "y": y})
             .assign(quintil=lambda d: pd.qcut(d["p"], q=5, duplicates="drop"))
             .groupby("quintil", observed=True)
             .agg(predicho=("p", "mean"), observado=("y", "mean"), n=("y", "size")))
    oe = y.mean() / p.mean()
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    logit = np.log(p_c / (1 - p_c))
    slope = LogisticRegression(C=1e6, max_iter=2000).fit(logit.reshape(-1, 1), y).coef_[0][0]
    results[name].update({"o_e_ratio": float(oe), "calibration_slope": float(slope)})
    print(f"\n{name}: O/E = {oe:.2f} | pendiente de calibración = {slope:.2f}")
    print(tab.round(3).to_string())

### Paso 11 — Desempeño por subgrupo (equidad)

TRIPOD+AI pide examinar si el modelo funciona igual en subgrupos relevantes.

**Explicación línea por línea**

- `for sexo, mask in [("mujeres", df["male"] == 0), ("hombres", df["male"] == 1)]:` dos subgrupos.
- `m = mask.to_numpy()`: convierte la máscara a arreglo NumPy para indexar `y` y `p`.
- `roc_auc_score(y[m], p[m])`: AUROC dentro del subgrupo; con pocas altas por grupo el resultado es inestable, por eso se informa también `n` y eventos.

In [ ]:
for name, p in oof.items():
    for sexo, mask in [("mujeres", df["male"] == 0), ("hombres", df["male"] == 1)]:
        m = mask.to_numpy()
        auc = roc_auc_score(y[m], p[m]) if y[m].min() != y[m].max() else float("nan")
        results[name][f"auroc_{sexo}"] = float(auc)
        print(f"{name:<9} {sexo:<8} n={m.sum():>4} eventos={int(y[m].sum()):>4} AUROC={auc:.3f}")

### Paso 12 — Modelo final, interpretación y guardado

**Explicación línea por línea**

- `final = MODELS["logistic"].fit(X, y)`: el modelo que se entrega se ajusta con **todos** los datos; la validación cruzada solo estimó cómo se comportará.
- `lr = final.named_steps["logisticregression"]`: accede al paso del pipeline por su nombre automático.
- `names = final.named_steps["simpleimputer"].get_feature_names_out(FEATURES)`: nombres de columnas tras el imputador (incluye las nuevas `missingindicator_*`).
- `np.exp(lr.coef_[0])`: *odds ratio* por cada desviación estándar del predictor (las variables están estandarizadas).
- `joblib.dump({...}, MODELS_DIR / "readmission_model.joblib")`: guarda el modelo **junto con** la lista de predictores y la versión de scikit-learn. **En la práctica real** un modelo sin su lista de variables y versiones no se puede volver a usar con seguridad.
- `json.dumps(results, indent=2)`: guarda todas las métricas para el reporte y para la semana 8.

In [ ]:
import joblib
import sklearn

final = MODELS["logistic"].fit(X, y)
lr = final.named_steps["logisticregression"]
names = final.named_steps["simpleimputer"].get_feature_names_out(FEATURES)
odds = pd.Series(np.exp(lr.coef_[0]), index=names).sort_values(ascending=False)
print("Odds ratio por 1 DE:")
print(odds.round(2).to_string())

joblib.dump({"model": final, "features": FEATURES, "sklearn_version": sklearn.__version__},
            MODELS_DIR / "readmission_model.joblib")
(REPORTS_DIR / "s07_metrics.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print("Guardado modelo y métricas")

### Paso 13 — Ficha del modelo con la estructura de TRIPOD+AI

TRIPOD+AI es una lista de 27 ítems para reportar estudios que desarrollan o evalúan modelos de predicción, con regresión o con *machine learning*. Esta ficha cubre los ítems centrales; el artículo completo debe seguir la lista oficial.

**Explicación línea por línea**

- `lg, gb = results["logistic"], results["gbm"]`: atajos.
- `card = f"""..."""`: texto Markdown con secciones: fuente de datos, participantes, desenlace, predictores, tamaño muestral, datos ausentes, análisis, desempeño, limitaciones y uso previsto. Las cifras se insertan desde `results` y `df`, así la ficha nunca contradice el análisis.
- `(REPORTS_DIR / "model_card_s07.md").write_text(card, encoding="utf-8")`: lo publica en tu Drive; súbelo también al repo (no contiene datos de pacientes).

In [ ]:
lg, gb = results["logistic"], results["gbm"]
card = f"""# Ficha del modelo — Reingreso o muerte a 30 días tras el alta (datos sintéticos)

## Fuente de datos
Synthea sample data CSV (MITRE), noviembre 2021, convertido a OMOP CDM v5.4 (semanas 1-6). Datos sintéticos: no representan pacientes reales.

## Participantes
Altas hospitalarias (visit_concept_id 9201) desde 2012-01-01, con alta en vida y 30 días de seguimiento posibles o evento.
Altas: {len(df)}; personas: {df['person_id'].nunique()}.

## Desenlace
Nueva hospitalización o muerte en los 30 días posteriores al alta (definición compuesta, como el índice LACE).
Eventos: {int(y.sum())} ({y.mean():.1%}).

## Predictores (medidos hasta el alta)
{", ".join(FEATURES)}. Comorbilidades definidas con conjuntos de conceptos SNOMED CT (semana 5).

## Tamaño muestral
Eventos por variable candidata: {y.sum() / len(FEATURES):.1f}.

## Datos ausentes
Imputación por mediana dentro de cada partición, con indicadores de ausencia.

## Análisis
Regresión logística (modelo final) y gradient boosting (comparador). Validación interna: validación cruzada estratificada agrupada por persona, 5 particiones × 5 repeticiones; IC95 % por bootstrap por paciente (300 réplicas).

## Desempeño (validación interna)
- Logística: AUROC {lg['auroc_mean']:.3f} (IC95 % {lg['auroc_ci95'][0]:.3f}–{lg['auroc_ci95'][1]:.3f}); Brier {lg['brier_mean']:.3f}; O/E {lg['o_e_ratio']:.2f}; pendiente de calibración {lg['calibration_slope']:.2f}.
- Gradient boosting: AUROC {gb['auroc_mean']:.3f} (IC95 % {gb['auroc_ci95'][0]:.3f}–{gb['auroc_ci95'][1]:.3f}); Brier {gb['brier_mean']:.3f}; O/E {gb['o_e_ratio']:.2f}; pendiente {gb['calibration_slope']:.2f}.
- Brier del modelo nulo: {brier_ref:.3f}.

## Limitaciones
1. Datos sintéticos: el modelo aprende las reglas del generador, no la fisiopatología.
2. Sin validación externa ni temporal.
3. No se distinguen reingresos programados de no programados.
4. Sin componente de urgencia del ingreso ("A" de LACE) ni índice de Charlson.
5. Pocas personas aportan muchas altas.

## Uso previsto
Exclusivamente educativo. No usar para decisiones clínicas.
"""
(REPORTS_DIR / "model_card_s07.md").write_text(card, encoding="utf-8")
print(card[:1200])
con.close()

### Autoevaluación

1. Da un ejemplo concreto de fuga temporal que podrías cometer con la tabla `measurement`.
2. ¿Por qué la validación ingenua infló el AUROC del modelo de árboles?
3. Un modelo con AUROC 0,80 y pendiente de calibración 0,6: ¿qué le pasa y qué harías?
4. ¿Por qué LACE combina reingreso y muerte en un único desenlace?

**Commit sugerido:** `feat(s07): modelo de reingreso a 30 días sobre OMOP con validación agrupada y ficha TRIPOD+AI`